# Filtering and Smoothing

We compare Gaussian and particle filters on state-space models:
1. **Linear Gaussian** — Kalman filter (exact), RTS smoother
2. **Nonlinear** — Extended Kalman filter, Unscented Kalman filter, Particle filter

All filters follow the same `Filter` runner API:

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

jax.config.update("jax_platform_name", "cpu")

from functools import partial

from probjax.inference.filter_smooth import Filter, filter_and_smooth
from probjax.inference.filtering.kalman_filter import kalman_filter
from probjax.inference.filtering.extended_kalman_filter import extended_kalman_filter
from probjax.inference.filtering.unscented_kalman_filter import ukf, merwe_sigma_point
from probjax.inference.filtering.particle_filter import ParticleFilter
from probjax.inference.filtering.smoothing import rauch_tung_stribel_smoother

## Linear Gaussian model

A simple 1D state-space model with known transition and observation dynamics:

$$x_{t+1} = A \, x_t + w_t, \quad w_t \sim \mathcal{N}(0, Q)$$
$$y_t = C \, x_t + v_t, \quad v_t \sim \mathcal{N}(0, R)$$

We generate synthetic data from the model, then filter and smooth.

In [ ]:
A = jnp.array([[0.95]])
Q = jnp.array([[0.1]])
C = jnp.array([[1.0]])
R = jnp.array([[0.5]])

mu0 = jnp.array([0.0])
cov0 = jnp.array([[1.0]])

# Transition and observation model callbacks
def transition_model(t_old, t):
    return A, Q

def observation_model(t):
    return C, R

# Generate synthetic observations
key = jax.random.PRNGKey(42)
T = 100
ts = jnp.arange(T, dtype=jnp.float32)

x_true = jnp.zeros((T, 1))
y_obs = jnp.zeros((T, 1))
x = mu0
for t in range(T):
    key, k1, k2 = jax.random.split(key, 3)
    x = A @ x + jax.random.normal(k1, (1, 1)) * jnp.sqrt(Q)
    y = C @ x + jax.random.normal(k2, (1, 1)) * jnp.sqrt(R)
    x_true = x_true.at[t].set(x[0])
    y_obs = y_obs.at[t].set(y[0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true[:, 0], label="True state", alpha=0.7)
ax.scatter(ts, y_obs[:, 0], s=8, color="red", alpha=0.4, label="Observations")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Linear Gaussian model — synthetic data")
plt.tight_layout(); plt.show()

## Kalman filter

The Kalman filter gives the exact posterior for linear Gaussian models.
We use the `Filter` runner class — analogous to `MCMC` / `SMC`.

In [ ]:
kernel = kalman_filter(transition_model, observation_model)
filt = Filter(kernel)

# Observations align with ts[1:] (scan starts at index 1)
t_o = ts[1:]   # observation times
x_o = y_obs[1:]  # observation values

trace = filt.filter(key, ts, t_o, x_o, mu0, cov0)

print(f"Trace contains {trace.states.mean.shape[0]} time steps")
print(f"State dim: {trace.states.mean.shape[1]}")
print(f"Filtered means shape:  {trace.states.mean.shape}")
print(f"Filtered covs shape:   {trace.states.cov.shape}")
print(f"Predicted means shape: {trace.infos.mean_pred.shape}")

In [ ]:
ts_filt = ts[1:]  # filtered outputs align with ts[1:]
mus = trace.states.mean[:, 0]
stds = jax.vmap(jnp.sqrt)(trace.states.cov[:, 0, 0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true[:, 0], alpha=0.5, label="True state")
ax.plot(ts_filt, mus, label="Filtered mean")
ax.fill_between(ts_filt, mus - 2*stds, mus + 2*stds, alpha=0.15, label="2σ")
ax.scatter(ts, y_obs[:, 0], s=6, color="red", alpha=0.3, label="Observations")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Kalman filter")
plt.tight_layout(); plt.show()

## Log-likelihood

The filter also exposes the log-likelihood of the observations under the model — useful for model selection and parameter learning.

In [ ]:
ll = filt.log_likelihood(key, ts, t_o, x_o, mu0, cov0)
print(f"Log-likelihood: {ll:.2f}")

## RTS smoothing

Smoothing refines the filtered estimates by incorporating future observations.
For Gaussian filters we pass an RTS-style smoother callback.

In [ ]:
rts_smoother = partial(rauch_tung_stribel_smoother, lambda t0, t1: A)

mus_s, covs_s = filt.smooth(trace, smoother=rts_smoother)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true[:, 0], alpha=0.5, label="True state")
ax.plot(ts_filt, trace.states.mean[:, 0], alpha=0.6, label="Filtered")
ax.plot(ts_filt, mus_s[:, 0], linewidth=2, label="Smoothed")
ax.scatter(ts_filt, x_o[:, 0], s=6, color="red", alpha=0.3, label="Observations")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Kalman filter + RTS smoother")
plt.tight_layout(); plt.show()

## Nonlinear model

When the transition or observation model is nonlinear, we use approximate filters.
Here we define a nonlinear model:

$$x_{t+1} = 0.5\, x_t + 25 \frac{x_t}{1 + x_t^2} + 8 \cos(1.2\, t) + w_t$$
$$y_t = \frac{x_t^2}{20} + v_t$$

In [ ]:
Q_nl = jnp.array([[10.0]])
R_nl = jnp.array([[1.0]])

def transition_fn_nl(x, t_old, t):
    return 0.5 * x + 25.0 * x / (1.0 + x**2) + 8.0 * jnp.cos(1.2 * t_old)

def observation_fn_nl(x, t):
    return x**2 / 20.0

# Jacobians for EKF (signature: (mu, cov, t) -> (matrix, covariance))
def transition_jac_nl(mu, cov, t):
    val = 0.5 + 25.0 * (1.0 - mu**2) / (1.0 + mu**2)**2
    return jnp.array([[val[0]]]), Q_nl

def observation_jac_nl(mu, cov, t):
    return jnp.array([[mu[0] / 10.0]]), R_nl

# Generate nonlinear data
key = jax.random.PRNGKey(7)
x_true_nl = jnp.zeros((T, 1))
y_obs_nl = jnp.zeros((T, 1))
x = jnp.array([0.0])
for t in range(T):
    key, k1, k2 = jax.random.split(key, 3)
    x = transition_fn_nl(x, t, t+1) + jax.random.normal(k1, (1,)) * jnp.sqrt(Q_nl[0, 0])
    y = observation_fn_nl(x, t+1) + jax.random.normal(k2, (1,)) * jnp.sqrt(R_nl[0, 0])
    x_true_nl = x_true_nl.at[t].set(x[0])
    y_obs_nl = y_obs_nl.at[t].set(y[0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true_nl[:, 0], alpha=0.7, label="True state")
ax.scatter(ts, y_obs_nl[:, 0], s=8, color="red", alpha=0.4, label="Observations")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Nonlinear model — synthetic data")
plt.tight_layout(); plt.show()

## Extended Kalman filter

The EKF linearizes the nonlinear model at each step via Jacobians.

In [ ]:
ekf_kernel = extended_kalman_filter(
    transition_fn_nl, observation_fn_nl,
    transition_jac_nl, observation_jac_nl,
)
filt_ekf = Filter(ekf_kernel)

t_o_nl = ts[1:]
x_o_nl = y_obs_nl[1:]
trace_ekf = filt_ekf.filter(key, ts, t_o_nl, x_o_nl, jnp.array([0.0]), jnp.array([[10.0]]))

mus_ekf = trace_ekf.states.mean[:, 0]
stds_ekf = jax.vmap(jnp.sqrt)(trace_ekf.states.cov[:, 0, 0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true_nl[:, 0], alpha=0.5, label="True state")
ax.plot(ts_filt, mus_ekf, label="EKF filtered")
ax.fill_between(ts_filt, mus_ekf - 2*stds_ekf, mus_ekf + 2*stds_ekf, alpha=0.15, label="2σ")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Extended Kalman filter")
plt.tight_layout(); plt.show()

## Unscented Kalman filter

The UKF propagates sigma points through the nonlinear model — no Jacobians needed.

In [ ]:
ukf_kernel = ukf(
    transition_fn_nl, Q_nl, observation_fn_nl, R_nl,
    sigma_point_fn=merwe_sigma_point,
)
filt_ukf = Filter(ukf_kernel)

trace_ukf = filt_ukf.filter(key, ts, t_o_nl, x_o_nl, jnp.array([0.0]), jnp.array([[10.0]]))

mus_ukf = trace_ukf.states.mean[:, 0]
stds_ukf = jax.vmap(jnp.sqrt)(trace_ukf.states.cov[:, 0, 0])

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true_nl[:, 0], alpha=0.5, label="True state")
ax.plot(ts_filt, mus_ukf, label="UKF filtered")
ax.fill_between(ts_filt, mus_ukf - 2*stds_ukf, mus_ukf + 2*stds_ukf, alpha=0.15, label="2σ")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Unscented Kalman filter")
plt.tight_layout(); plt.show()

## Particle filter

The particle filter is fully nonparametric — it represents the posterior as a
weighted cloud of particles.  No linearization or Gaussian assumptions.

In [ ]:
num_particles = 256

def pf_transition(key, particles, t):
    noise = jax.random.normal(key, particles.shape) * jnp.sqrt(Q_nl[0, 0])
    return jax.vmap(lambda x: transition_fn_nl(x, t, t+1))(particles) + noise

def pf_log_likelihood(particles, obs, t):
    y_pred = jax.vmap(lambda x: observation_fn_nl(x, t))(particles)[:, 0]
    return jax.scipy.stats.norm.logpdf(obs[0], y_pred, jnp.sqrt(R_nl[0, 0]))

pf_kernel = ParticleFilter(pf_log_likelihood, pf_transition)
filt_pf = Filter(pf_kernel)

key = jax.random.PRNGKey(0)
init_particles = jax.random.normal(key, (num_particles, 1)) * 3.0

trace_pf = filt_pf.filter(key, ts, t_o_nl, x_o_nl, init_particles)

pf_mean = trace_pf.states.particles.mean(axis=1)
pf_std = trace_pf.states.particles.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts, x_true_nl[:, 0], alpha=0.7, linewidth=2, label="True state")
ax.plot(ts_filt, pf_mean[:, 0], label="PF mean")
ax.fill_between(ts_filt, pf_mean[:, 0] - 2*pf_std[:, 0], pf_mean[:, 0] + 2*pf_std[:, 0], alpha=0.15, label="2σ")
ax.set_xlabel("Time"); ax.set_ylabel("x"); ax.legend()
ax.set_title("Particle filter (256 particles)")
plt.tight_layout(); plt.show()

## Particle smoothing

For particle filters, smoothing uses the FFBSi (Forward Filter-Backward Simulator)
algorithm.  We pass a `transition_logdensity_fn` that evaluates $\log p(x_{t+1} \mid x_t)$.

In [ ]:
def transition_logdensity(x_tp1, x_t, t, tp1):
    mean = transition_fn_nl(x_t, t, tp1)
    return jax.scipy.stats.norm.logpdf(x_tp1, mean, jnp.sqrt(Q_nl[0, 0])).sum()

smoothed_particles, smoothed_log_weights = filt_pf.smooth(
    trace_pf,
    key=jax.random.PRNGKey(1),
    transition_logdensity_fn=transition_logdensity,
)

sm_mean = smoothed_particles.mean(axis=1)
sm_std = smoothed_particles.std(axis=1)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(ts, x_true_nl[:, 0], alpha=0.5, label="True state")
axes[0].plot(ts_filt, pf_mean[:, 0], alpha=0.6, label="Filtered mean")
axes[0].set_ylabel("x"); axes[0].legend()
axes[0].set_title("Particle filter (filtered)")

axes[1].plot(ts, x_true_nl[:, 0], alpha=0.5, label="True state")
axes[1].plot(ts_filt, sm_mean[:, 0], linewidth=2, label="Smoothed mean")
axes[1].fill_between(ts_filt, sm_mean[:, 0] - 2*sm_std[:, 0], sm_mean[:, 0] + 2*sm_std[:, 0], alpha=0.15, label="2σ")
axes[1].set_xlabel("Time"); axes[1].set_ylabel("x"); axes[1].legend()
axes[1].set_title("Particle smoother (smoothed)")

plt.tight_layout(); plt.show()

## Comparing filters

All three nonlinear filters can be compared side by side on the same model.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, (name, mus, stds) in zip(axes, [
    ("EKF", mus_ekf, stds_ekf),
    ("UKF", mus_ukf, stds_ukf),
    ("PF (256)", pf_mean[:, 0], pf_std[:, 0]),
]):
    ax.plot(ts, x_true_nl[:, 0], alpha=0.5, label="True state")
    ax.plot(ts_filt, mus, label=f"{name} filtered")
    ax.fill_between(ts_filt, mus - 2*stds, mus + 2*stds, alpha=0.15, label="2σ")
    ax.scatter(ts, y_obs_nl[:, 0], s=4, color="red", alpha=0.3)
    ax.set_ylabel("x"); ax.legend()
    ax.set_title(name)

axes[-1].set_xlabel("Time")
plt.tight_layout(); plt.show()

## Verbose mode

Like `MCMC(kernel, verbose=True)`, the `Filter` runner can display a progress bar
with running log-likelihood (useful for long time series).

In [ ]:
filt_verbose = Filter(kalman_filter(transition_model, observation_model), verbose=True)

# Run a longer series to see the progress bar
ts_long = jnp.arange(2000, dtype=jnp.float32)
y_long = jnp.zeros((2000, 1))  # placeholder

t_o_long = ts_long[1:]
x_o_long = y_long[1:]
_ = filt_verbose.filter(key, ts_long, t_o_long, x_o_long, mu0, cov0)

## `filter_and_smooth` convenience

For one-shot filtering + smoothing, use the `filter_and_smooth` function.

In [ ]:
trace, (mus_s2, covs_s2) = filter_and_smooth(
    key, ts, t_o, x_o,
    kalman_filter(transition_model, observation_model),
    mu0, cov0,
    smoother=rts_smoother,
)

print(f"Smoothed means shape:  {mus_s2.shape}")
print(f"Smoothed covs shape:   {covs_s2.shape}")

## Observations

**Linear model:** The Kalman filter is exact — no approximation error. RTS smoothing refines estimates using future observations, producing tighter uncertainty bands.

**Nonlinear model:**
- **EKF** linearizes via Jacobians — fast but can diverge on highly nonlinear models.
- **UKF** propagates sigma points — more robust than EKF, no Jacobians needed.
- **Particle filter** is nonparametric — handles arbitrary nonlinearities but requires more particles for high-dimensional states.

**API design:** The `Filter` runner follows the same pattern as `MCMC` and `SMC` —
construct with a kernel, then call `.filter()`, `.smooth()`, `.log_likelihood()`.  This
makes it easy to swap filters without changing orchestration code.